# C1 · Datos biológicos, FAIR y organización de proyectos

**Curso:** Bioinformática y Biología Computacional · Universidad EAFIT  
**Duración sugerida:** 3 horas  
**Modalidad:** explicación breve → práctica guiada → reto → evidencia reproducible

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UniversidadEAFIT/compubiol_course/blob/master/notebooks/01_datos_fair/01_datos_fair.ipynb)

## Pregunta guía

Un proyecto recibe secuencias, anotaciones y metadatos de varios colaboradores. Los nombres no coinciden, faltan campos y nadie sabe qué archivo es la versión final. **¿Qué debe ocurrir antes de iniciar un análisis?**

### Objetivos

- distinguir FASTA, FASTQ, GFF3, VCF y metadatos tabulares;
- inventariar datos sin modificar los originales;
- detectar inconsistencias de identificadores, fechas, categorías y procedencia;
- aplicar los principios FAIR como decisiones concretas;
- proponer una estructura y un diccionario mínimo de datos.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import importlib.util

REPO_URL = "https://github.com/UniversidadEAFIT/compubiol_course.git"
COLAB_DIR = Path("/content/compubiol_course")

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB and importlib.util.find_spec("Bio") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

if IN_COLAB and not COLAB_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(COLAB_DIR)], check=True)
    os.chdir(COLAB_DIR)

start = Path.cwd().resolve()
ROOT = next((p for p in [start, *start.parents] if (p / "data").is_dir() and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError(
        "No se encontró la raíz del curso. Ejecute el notebook desde el repositorio clonado."
    )
os.chdir(ROOT)
os.environ["COURSE_ROOT"] = str(ROOT)
print(f"Raíz del curso: {ROOT}")

## 1. De “archivos” a activos de investigación

Un archivo es reutilizable cuando tiene contexto: procedencia, versión, formato, unidades, relaciones con otros archivos y reglas para valores faltantes. FAIR no significa que todo deba ser público: significa que los datos sean **localizables, accesibles bajo condiciones explícitas, interoperables y reutilizables**.

| Formato | Unidad principal | Pregunta de control |
|---|---|---|
| FASTA | secuencia | ¿qué representa cada encabezado? |
| FASTQ | lectura + calidad | ¿qué codificación y plataforma? |
| GFF3/GTF | característica genómica | ¿qué sistema de coordenadas y relaciones ID/Parent? |
| VCF | variante | ¿contra qué referencia y con qué filtros? |
| TSV/CSV | metadatos | ¿qué significa cada columna y categoría? |

In [ ]:
from pathlib import Path

files = sorted((ROOT / "data/module01").glob("*"))
for path in files:
    if path.is_file():
        print(f"{path.name:32s} {path.stat().st_size:6d} bytes")

## 2. Inspeccionar sin sobrescribir

In [ ]:
import pandas as pd

metadata_path = ROOT / "data/module01/metadata_inconsistent.tsv"
metadata = pd.read_csv(metadata_path, sep="\t", dtype=str, keep_default_na=False)
metadata

In [ ]:
checks = pd.DataFrame({
    "sample_id": metadata["sample_id"],
    "normalized_id": (
        metadata["sample_id"].str.upper().str.replace("_", "-", regex=False)
    ),
    "missing_country": metadata["country"].eq(""),
    "missing_replicate": metadata["replicate"].eq(""),
    "file_exists": metadata["sequence_file"].map(
        lambda x: (ROOT / "data/module01" / x).exists()
    ),
})
checks["duplicate_after_normalization"] = checks["normalized_id"].duplicated(keep=False)
checks

### Checkpoint 1

Explique por qué `S-001` y `s001` pueden ser un duplicado lógico aun cuando sus cadenas no sean idénticas. No elimine ninguna fila hasta definir una regla y conservar un registro de la decisión.

In [ ]:
from datetime import datetime

formats = ["%Y-%m-%d", "%d/%m/%Y", "%Y/%m/%d"]
def parse_date(value: str):
    for fmt in formats:
        try:
            return datetime.strptime(value, fmt).date().isoformat()
        except ValueError:
            pass
    return None

metadata.assign(parsed_date=metadata["collection_date"].map(parse_date))[
    ["sample_id", "collection_date", "parsed_date"]
]

## 3. Estructura de proyecto

Una estructura sencilla evita mezclar entradas, código y resultados:

```text
project/
├── README.md
├── data/
│   ├── raw/          # solo lectura
│   ├── interim/      # transformaciones reversibles
│   └── processed/    # datos listos para análisis
├── metadata/
├── scripts/
├── notebooks/
├── results/
├── logs/
└── environment.yml
```

Los datos `raw/` no se corrigen “en sitio”. Cada transformación debe poder reconstruirse mediante código.

In [ ]:
project = ROOT / "results/module01/project_skeleton"
for rel in ["data/raw", "data/interim", "data/processed", "metadata", "scripts", "notebooks", "results", "logs"]:
    (project / rel).mkdir(parents=True, exist_ok=True)
print("\n".join(str(p.relative_to(project)) for p in sorted(project.rglob("*"))))

## 4. Mini auditoría FAIR

Califique de 0 a 2 cada dimensión para el conjunto recibido:

- **F — Findable:** identificadores, catálogo, versión y descripción.
- **A — Accessible:** protocolo y condiciones de acceso; no exige acceso abierto.
- **I — Interoperable:** formatos documentados, vocabularios y unidades consistentes.
- **R — Reusable:** licencia, procedencia, calidad, transformaciones y contexto.

### Reto

Produzca una tabla curada, un diccionario y un plan de gestión. Incluya una columna `issue_log` o un archivo separado con la regla aplicada a cada problema.

## Cierre

**Evidencia mínima:** inventario, tabla original intacta, tabla curada, diccionario, estructura de proyecto y decisiones FAIR.  
**Error que debe evitarse:** empezar BLAST o cualquier análisis antes de saber qué representa cada muestra.

Referencias: Wilkinson et al. (2016); Data Carpentry Genomics; documentación de formatos de NCBI.